In [1]:
import requests
import pandas as pd
import random

def extract_stratified_users(users_per_bracket=250):
    print("Fetching users from codeforces API!")
    url = "https://codeforces.com/api/user.ratedList?activeOnly=true"
    
    try:
        response = requests.get(url).json()
        if response.get('status') != 'OK':
            print("API Error fetching users.")
            return
    except Exception as e:
        print(f"Connection Error: {e}")
        return

    all_users = response['result']
    print(f" Found {len(all_users)} total active users.")
    
    # Official title Brackets!
    brackets = {
        "Newbie (<1200)": [],
        "Pupil (1200-1399)": [],
        "Specialist (1400-1599)": [],
        "Expert (1600-1899)": [],
        "Candidate Master (1900-2099)": [],
        "Master & Above (>=2100)": []
    }
    
    for u in all_users:
        rating = u.get('rating', 0)
        handle = u['handle']
        
        # Create a clean dictionary for each user
        user_data = {'handle': handle, 'rating': rating}
        
        if rating < 1200:
            user_data['bracket'] = "Newbie (<1200)"
            brackets["Newbie (<1200)"].append(user_data)
            
        elif rating < 1400:
            user_data['bracket'] = "Pupil (1200-1399)"
            brackets["Pupil (1200-1399)"].append(user_data)
            
        elif rating < 1600:
            user_data['bracket'] = "Specialist (1400-1599)"
            brackets["Specialist (1400-1599)"].append(user_data)
            
        elif rating < 1900:
            user_data['bracket'] = "Expert (1600-1899)"
            brackets["Expert (1600-1899)"].append(user_data)
            
        elif rating < 2100:
            user_data['bracket'] = "Candidate Master (1900-2099)"
            brackets["Candidate Master (1900-2099)"].append(user_data)
            
        else:
            user_data['bracket'] = "Master & Above (>=2100)"
            brackets["Master & Above (>=2100)"].append(user_data)

    final_cohort = []
    print("\n--- Sampling Results ---")
    
    for bracket_name, users_in_bracket in brackets.items():
        # If a bracket has fewer users than requested (rare, but possible), take all of them
        sample_size = min(users_per_bracket, len(users_in_bracket))
        sampled = random.sample(users_in_bracket, sample_size)
        
        print(f" {bracket_name}: Selected {sample_size} users")
        final_cohort.extend(sampled)
            
    # 4. Save directly to CSV
    df = pd.DataFrame(final_cohort)
    filename = "cf_stratified_users.csv"
    df.to_csv(filename, index=False)
    
    print(f"\n Success! Saved {len(df)} balanced users to '{filename}'.")


In [2]:
extract_stratified_users(300)

Fetching users from codeforces API!
 Found 50484 total active users.

--- Sampling Results ---
 Newbie (<1200): Selected 300 users
 Pupil (1200-1399): Selected 300 users
 Specialist (1400-1599): Selected 300 users
 Expert (1600-1899): Selected 300 users
 Candidate Master (1900-2099): Selected 300 users
 Master & Above (>=2100): Selected 300 users

 Success! Saved 1800 balanced users to 'cf_stratified_users.csv'.


In [3]:
#Scraping submissions per user!
import requests
import pandas as pd
import time
import os

def mine_submissions_from_csv():
    input_file = "cf_stratified_users.csv"
    output_file = "cf_submissions.csv"
    
    print(f"[*] Reading user cohort from '{input_file}'...")
    try:
        df_users = pd.read_csv(input_file)
        all_handles = df_users['handle'].tolist()
    except FileNotFoundError:
        print(f"[-] Error: Could not find '{input_file}'. Run the extraction script first.")
        return

    # Check if we already have partial progress
    completed_handles = set()
    if os.path.exists(output_file):
        try:
            existing_df = pd.read_csv(output_file, usecols=['handle'])
            completed_handles = set(existing_df['handle'].unique())
            print(f"[*] Found existing progress. Skipping {len(completed_handles)} users already mined.")
        except Exception as e:
            print(f"[*] Starting fresh file (could not read existing progress).")

    handles_to_process = [h for h in all_handles if h not in completed_handles]
    
    print(f"\n[*] Starting data extraction for {len(handles_to_process)} users...")
    print(f"[!] ETA: ~{int((len(handles_to_process) * 1.5) / 60)} minutes. Grab a coffee and let it run!")
    
    for idx, handle in enumerate(handles_to_process):
        if idx % 10 == 0 and idx > 0:
            print(f"    -> Progress: {idx}/{len(handles_to_process)} users processed...")
            
        url = f"https://codeforces.com/api/user.status?handle={handle}&from=1&count=500"
        
        try:
            response = requests.get(url).json()
            if response.get('status') == 'OK':
                submissions = response['result']
                user_subs_data = []
                
                for sub in submissions:
                    # We join tags with a pipe "|" so it doesn't break CSV formatting
                    tags_list = sub['problem'].get('tags', [])
                    tags_str = "|".join(tags_list) 
                    
                    user_subs_data.append({
                        'submission_id': sub['id'],
                        'handle': handle,
                        'problem_id': f"{sub['problem'].get('contestId', '')}{sub['problem'].get('index', '')}",
                        'rating': sub['problem'].get('rating', 0), # Getting the rating of the problem
                        'verdict': sub.get('verdict', 'UNKNOWN'),
                        'time_consumed_ms': sub.get('timeConsumedMillis', 0),
                        'tags': tags_str,
                        'creation_time': sub.get('creationTimeSeconds', 0) # Useful for finding "Tilt Factor"
                    })
                
                if user_subs_data:
                    df_subs = pd.DataFrame(user_subs_data)
                    write_header = not os.path.exists(output_file)
                    df_subs.to_csv(output_file, mode='a', index=False, header=write_header)
                
            else:
                print(f"    [!] Skipping {handle}: {response.get('comment')}")
                
        except Exception as e:
            print(f"    [-] Error fetching {handle}: {e}")
            
        time.sleep(1.5) 

    print("\n[+] Extraction Complete! All submissions are safely stored in 'cf_submissions.csv'.")


In [5]:
mine_submissions_from_csv()

[*] Reading user cohort from 'cf_stratified_users.csv'...
[*] Found existing progress. Skipping 1509 users already mined.

[*] Starting data extraction for 291 users...
[!] ETA: ~7 minutes. Grab a coffee and let it run!
    [-] Error fetching yashkale_alt: Expecting value: line 1 column 1 (char 0)
    [-] Error fetching XT_Abhi: Expecting value: line 1 column 1 (char 0)
    [-] Error fetching sujalagrawal: Expecting value: line 1 column 1 (char 0)
    [-] Error fetching _Karamelka_: Expecting value: line 1 column 1 (char 0)
    [-] Error fetching Hushpuppy: Expecting value: line 1 column 1 (char 0)
    [-] Error fetching Anemone_: Expecting value: line 1 column 1 (char 0)
    [-] Error fetching solver777: Expecting value: line 1 column 1 (char 0)
    [-] Error fetching Enoch006: Expecting value: line 1 column 1 (char 0)
    [-] Error fetching ItsNotMeItsYou: Expecting value: line 1 column 1 (char 0)
    [-] Error fetching MR_NoSolution: Expecting value: line 1 column 1 (char 0)
    -> 